# Video 2 — Dose-Response Analysis and IC50 Fitting
**AI for Drug Discovery series**  
Author: Sreenivas Bhattiprolu (DigitalSreeni)  
YouTube: [youtube.com/@DigitalSreeni](https://www.youtube.com/@DigitalSreeni)  
GitHub: [github.com/bnsreenu](https://github.com/bnsreenu)

---

## What this notebook covers

1. The 4-parameter logistic (4PL) model and what each parameter means
2. Fitting dose-response curves and extracting IC50 with confidence intervals
3. Numerical quality metrics: R-squared, residual error, CI width
4. LLM-based curve quality assessment using Claude Sonnet
5. A brief pointer to real dose-response data from ChEMBL

**Data:** Synthetic curves generated by `video2_00_synthesize_data.py`  
**Environment:** Google Colab (CPU only, no GPU required)

## 1. Mount Google Drive and load API keys

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

ENV_PATH = (
    '/content/drive/MyDrive/ColabNotebooks/AI_for_drug_discovery/llm_keys/.env'
)

with open(ENV_PATH) as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            key, value = line.split('=', 1)
            os.environ[key.strip()] = value.strip()

OPENAI_API_KEY    = os.environ.get('OPENAI_API_KEY')
ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY')

print(f'OpenAI key loaded:    {OPENAI_API_KEY[:8]}...'    if OPENAI_API_KEY    else 'OpenAI key NOT found')
print(f'Anthropic key loaded: {ANTHROPIC_API_KEY[:8]}...' if ANTHROPIC_API_KEY else 'Anthropic key NOT found')

## 2. Install and import libraries

In [ ]:
# anthropic is not pre-installed on Colab
!pip install anthropic -q

# Optional: uncomment if using OpenAI
# !pip install openai -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from scipy.optimize import curve_fit
from scipy.stats import t as t_dist
import warnings
import base64
import io
import anthropic

# Uncomment if using OpenAI
# from openai import OpenAI

# Consistent plot style across all figures
plt.rcParams.update({
    'font.family':      'DejaVu Sans',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'figure.dpi':       120,
})

print('All libraries imported.')

## 3. Set data paths

In [ ]:
DATA_DIR = (
    '/content/drive/MyDrive/ColabNotebooks/AI_for_drug_discovery/Video2_dose_response_curves/data'
)

# Compound files produced by video2_00_synthesize_data.py
COMPOUND_FILES = {
    'Compound A (clean)':      f'{DATA_DIR}/compound_A_clean.csv',
    'Compound B (noisy)':      f'{DATA_DIR}/compound_B_noisy.csv',
    'Compound C (hook)':       f'{DATA_DIR}/compound_C_hook.csv',
    'Compound D (incomplete)': f'{DATA_DIR}/compound_D_incomplete.csv',
    'Compound E (inactive)':   f'{DATA_DIR}/compound_E_inactive.csv',
}

print('Data paths configured.')

## 4. The 4-parameter logistic (4PL) model

The 4PL model is the industry standard for fitting dose-response curves:

$$
\text{Response} = \text{Bottom} + \frac{\text{Top} - \text{Bottom}}{1 + \left(\frac{\text{IC}_{50}}{\text{concentration}}\right)^{\text{Hill}}}
$$

| Parameter | Meaning | Typical value |
|-----------|---------|---------------|
| Bottom    | Response at saturating compound concentration | ~0 (full inhibition) |
| Top       | Response with no compound (vehicle control) | ~100% viability |
| IC50      | Concentration producing 50% of maximal effect | depends on compound |
| Hill      | Slope of the sigmoid (cooperativity) | 0.5 to 3 in practice |

The cell below defines the model and plots a clean example so you can see
how each parameter shapes the curve.

In [ ]:
def four_pl(concentration, bottom, top, ic50, hill):
    """4-parameter logistic dose-response model."""
    return bottom + (top - bottom) / (1.0 + (ic50 / concentration) ** hill)


# --- Visualise the effect of each parameter ---
conc_range = np.logspace(-3, 2, 300)   # 0.001 to 100 uM

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# Vary IC50
ax = axes[0]
for ic50, color in zip([0.1, 1.0, 10.0], ['#0D9488', '#1B2A4A', '#E57373']):
    ax.semilogx(conc_range,
                four_pl(conc_range, 100, 0, ic50, 1.5),
                color=color, lw=2, label=f'IC50 = {ic50} uM')
ax.set(xlabel='Concentration (uM)', ylabel='Viability (%)',
       title='Effect of IC50')
ax.legend(fontsize=9)

# Vary Hill slope
ax = axes[1]
for hill, color in zip([0.5, 1.5, 3.0], ['#0D9488', '#1B2A4A', '#E57373']):
    ax.semilogx(conc_range,
                four_pl(conc_range, 100, 0, 1.0, hill),
                color=color, lw=2, label=f'Hill = {hill}')
ax.set(xlabel='Concentration (uM)', ylabel='Viability (%)',
       title='Effect of Hill slope')
ax.legend(fontsize=9)

# Vary Bottom (incomplete inhibition)
# Bottom is the high-concentration floor (residual viability).
# Curve starts at 100% (no effect) and falls to the Bottom value.
# Use (100, floor) so the curve goes downward: low conc -> 100%, high conc -> floor.
ax = axes[2]
for floor, color in zip([0, 30, 60], ['#0D9488', '#1B2A4A', '#E57373']):
    ax.semilogx(conc_range,
                four_pl(conc_range, 100, floor, 1.0, 1.5),
                color=color, lw=2, label=f'Bottom = {floor}%')
ax.set(xlabel='Concentration (uM)', ylabel='Viability (%)',
       title='Effect of Bottom plateau')
ax.legend(fontsize=9)

fig.suptitle('4PL Model: Parameter Effects', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Load synthetic data and visualise all five compounds

In [ ]:
# Load all compound CSVs into a dictionary
data = {}
for name, path in COMPOUND_FILES.items():
    df = pd.read_csv(path)
    data[name] = df
    print(f'{name}: {len(df)} concentration points')

print('\nSample from Compound A:')
data['Compound A (clean)'].head()

In [ ]:
# Raw data overview -- all five compounds on one figure
fig, axes = plt.subplots(1, 5, figsize=(17, 4), sharey=True)
colors = ['#0D9488', '#1B2A4A', '#E57373', '#F59E0B', '#6B7280']

for ax, (name, df), color in zip(axes, data.items(), colors):
    ax.semilogx(df['concentration_uM'], df['response_pct'],
                'o', color=color, ms=6, alpha=0.8)
    ax.set(xlabel='Concentration (uM)', title=name.split('(')[0].strip(),
           ylim=(-10, 120))
    ax.axhline(50, color='gray', lw=0.8, ls='--', alpha=0.5)

axes[0].set_ylabel('Response (%)')
fig.suptitle('Raw Dose-Response Data -- Five Compound Scenarios',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Curve fitting and IC50 extraction

We use `scipy.optimize.curve_fit`, which fits the 4PL model using nonlinear
least squares. The key challenge is providing good initial parameter guesses
(`p0`) -- poor guesses cause the optimiser to converge to a wrong solution
or fail entirely.

Our initialisation strategy:
- **Bottom**: minimum observed response
- **Top**: maximum observed response  
- **IC50**: geometric mean of the concentration range (midpoint on log scale)
- **Hill**: start at 1 (neutral slope)

In [ ]:
def make_initial_guess(conc, resp):
    """Data-driven initial parameter guess for 4PL fitting."""
    bottom = float(np.percentile(resp, 5))    # robust minimum
    top    = float(np.percentile(resp, 95))   # robust maximum
    ic50   = float(np.exp(np.mean(np.log(conc))))  # geometric mean
    hill   = 1.0
    return [bottom, top, ic50, hill]


def fit_curve(conc, resp):
    """
    Fit a 4PL model to concentration-response data.

    Returns
    -------
    dict with keys:
        params      -- fitted [bottom, top, ic50, hill]
        pcov        -- parameter covariance matrix
        ic50        -- fitted IC50 (uM)
        ic50_ci     -- (lower, upper) 95% CI on IC50
        r_squared   -- coefficient of determination
        rmse        -- root mean squared error
        success     -- bool
        message     -- status string
    """
    result = dict(params=None, pcov=None, ic50=None, ic50_ci=(None, None),
                  r_squared=None, rmse=None, success=False, message='')

    conc = np.asarray(conc, dtype=float)
    resp = np.asarray(resp, dtype=float)

    p0     = make_initial_guess(conc, resp)
    #bounds = ([0, 0, conc.min()*0.01, 0.1],
    bounds = ([-20, -20, conc.min()*0.01, 0.1],
              [150, 150, conc.max()*100, 10])

    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            popt, pcov = curve_fit(
                four_pl, conc, resp,
                p0=p0, bounds=bounds,
                maxfev=10000,
            )
    except Exception as e:
        result['message'] = f'Fitting failed: {e}'
        return result

    bottom, top, ic50, hill = popt

    # R-squared
    resp_pred = four_pl(conc, *popt)
    ss_res    = np.sum((resp - resp_pred) ** 2)
    ss_tot    = np.sum((resp - resp.mean()) ** 2)
    r2        = 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0

    # RMSE
    rmse = np.sqrt(ss_res / len(resp))

    # 95% confidence interval on IC50
    perr   = np.sqrt(np.diag(pcov))   # parameter standard errors
    ic50_idx = 2
    df_resid = max(len(resp) - 4, 1)
    t_crit   = t_dist.ppf(0.975, df=df_resid)
    ic50_lo  = ic50 - t_crit * perr[ic50_idx]
    ic50_hi  = ic50 + t_crit * perr[ic50_idx]

    result.update({
        'params':    popt,
        'pcov':      pcov,
        'ic50':      ic50,
        'ic50_ci':   (max(ic50_lo, 0), ic50_hi),
        'r_squared': r2,
        'rmse':      rmse,
        'success':   True,
        'message':   'OK',
    })
    return result


print('Fitting functions defined.')

In [ ]:
# Fit all five compounds and collect results
fit_results = {}

for name, df in data.items():
    conc = df['concentration_uM'].values
    resp = df['response_pct'].values
    fit_results[name] = fit_curve(conc, resp)
    r = fit_results[name]
    if r['success']:
        print(f"{name}")
        print(f"  IC50      = {r['ic50']:.3f} uM  "
              f"(95% CI: {r['ic50_ci'][0]:.3f} -- {r['ic50_ci'][1]:.3f})")
        print(f"  R-squared = {r['r_squared']:.3f}   RMSE = {r['rmse']:.2f}%")
        p = r['params']
        print(f"  Bottom={p[0]:.1f}  Top={p[1]:.1f}  Hill={p[3]:.2f}")
    else:
        print(f"{name}: {r['message']}")
    print()

## 7. Visualise fitted curves

In [ ]:
def plot_fitted_curve(ax, name, df, fit, color):
    """Plot raw data and fitted 4PL curve on a given axes."""
    conc = df['concentration_uM'].values
    resp = df['response_pct'].values

    # Raw data
    ax.semilogx(conc, resp, 'o', color=color, ms=7, alpha=0.85,
                zorder=3, label='Data')

    if fit['success']:
        conc_fine = np.logspace(
            np.log10(conc.min()) - 0.3,
            np.log10(conc.max()) + 0.3,
            300
        )
        resp_fit = four_pl(conc_fine, *fit['params'])
        ax.semilogx(conc_fine, resp_fit, '-', color=color, lw=2.5,
                    alpha=0.9, label='4PL fit')

        # IC50 marker
        ic50 = fit['ic50']
        resp_at_ic50 = four_pl(np.array([ic50]), *fit['params'])[0]
        ax.axvline(ic50, color='gray', lw=1, ls='--', alpha=0.6)
        ax.axhline(resp_at_ic50, color='gray', lw=1, ls='--', alpha=0.6)
        ax.annotate(
            f'IC50\n{ic50:.2f} uM',
            xy=(ic50, resp_at_ic50),
            xytext=(ic50 * 3, resp_at_ic50 + 15),
            fontsize=8, color='gray',
            arrowprops=dict(arrowstyle='->', color='gray', lw=0.8),
        )

        # Quality summary in corner
        ax.text(0.03, 0.08,
                f"R²={fit['r_squared']:.2f}  RMSE={fit['rmse']:.1f}%",
                transform=ax.transAxes, fontsize=8, color='#444')
    else:
        ax.text(0.5, 0.5, 'Fit failed', transform=ax.transAxes,
                ha='center', color='red', fontsize=10)

    short = name.split('(')[0].strip()
    scenario = name[name.find('(')+1:name.find(')')]
    ax.set_title(f'{short}\n({scenario})', fontsize=10, fontweight='bold')
    ax.set(xlabel='Concentration (uM)', ylim=(-15, 125))
    ax.axhline(50, color='lightgray', lw=0.8, ls=':')


fig, axes = plt.subplots(1, 5, figsize=(17, 4), sharey=True)
colors = ['#0D9488', '#1B2A4A', '#E57373', '#F59E0B', '#6B7280']

for ax, (name, df), color in zip(axes, data.items(), colors):
    plot_fitted_curve(ax, name, df, fit_results[name], color)

axes[0].set_ylabel('Response (%)')
fig.suptitle('4PL Curve Fitting -- Five Scenarios',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Quality metrics summary table

In [ ]:
# True IC50 values from the data generator (ground truth)
TRUE_IC50 = {
    'Compound A (clean)':      1.0,
    'Compound B (noisy)':      1.0,
    'Compound C (hook)':       0.5,   # meaningful only below 10 uM
    'Compound D (incomplete)': 50.0,  # outside tested range
    'Compound E (inactive)':   None,
}

rows = []
for name, fit in fit_results.items():
    true_ic50 = TRUE_IC50[name]
    if fit['success']:
        ic50     = fit['ic50']
        ci_lo, ci_hi = fit['ic50_ci']
        ci_width = ci_hi - ci_lo
        error    = abs(ic50 - true_ic50) / true_ic50 * 100 if true_ic50 else None
        rows.append({
            'Compound':       name,
            'True IC50 (uM)': true_ic50,
            'Fitted IC50 (uM)': round(ic50, 3),
            '95% CI lower':   round(ci_lo, 3),
            '95% CI upper':   round(ci_hi, 3),
            'CI width':       round(ci_width, 3),
            'R-squared':      round(fit['r_squared'], 3),
            'RMSE (%)':       round(fit['rmse'], 2),
            '% error vs true':round(error, 1) if error is not None else 'N/A',
        })
    else:
        rows.append({'Compound': name, 'True IC50 (uM)': true_ic50,
                     'Fitted IC50 (uM)': 'FAILED'})

summary = pd.DataFrame(rows)
summary

## 9. LLM-based curve quality assessment

A fitted IC50 number alone does not tell you whether the result is trustworthy.
An experienced scientist looks at the curve shape and asks questions like:

- Does the curve show a complete sigmoid (both plateaus visible)?
- Is there a hook effect at high concentration?
- Is the scatter reasonable or is the noise overwhelming the signal?
- How confident should I be in the IC50?

In this section we ask Claude Sonnet to do the same assessment from the curve image.

In [ ]:
def render_curve_for_llm(name, df, fit, color):
    """
    Render a clean single-curve plot and return it as a base64-encoded PNG.
    The image is not saved to disk -- it is held in memory for the API call.
    """
    fig, ax = plt.subplots(figsize=(5, 4))
    conc = df['concentration_uM'].values
    resp = df['response_pct'].values

    ax.semilogx(conc, resp, 'o', color=color, ms=8, alpha=0.85, label='Data')

    if fit['success']:
        conc_fine = np.logspace(
            np.log10(conc.min()) - 0.3,
            np.log10(conc.max()) + 0.3, 300
        )
        ax.semilogx(conc_fine, four_pl(conc_fine, *fit['params']),
                    '-', color=color, lw=2.5, label='4PL fit')
        ax.set_title(
            f"{name}  |  IC50 = {fit['ic50']:.2f} uM  "
            f"R² = {fit['r_squared']:.2f}",
            fontsize=10
        )
    else:
        ax.set_title(f"{name}  |  fit failed", fontsize=10)

    ax.set(xlabel='Concentration (uM)', ylabel='Viability (%)',
           ylim=(-15, 125))
    ax.axhline(50, color='lightgray', lw=0.8, ls=':')
    ax.legend(fontsize=9)
    plt.tight_layout()

    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=150, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    return base64.standard_b64encode(buf.read()).decode('utf-8')


print('Image rendering function defined.')

In [ ]:
# ── CLAUDE SONNET ──────────────────────────────────────────────────────────

def assess_curve_claude(name, df, fit, color, client):
    """
    Send a dose-response curve image to Claude Sonnet and request a
    structured quality assessment.
    """
    img_b64 = render_curve_for_llm(name, df, fit, color)

    prompt = """You are a medicinal chemist reviewing dose-response curves from a cell viability drug screening assay.

This is a VIABILITY assay. The y-axis shows cell viability (%). A well-behaved curve starts
near 100% viability at low concentrations (no effect) and falls to near 0% at high
concentrations (full inhibition). The curve should slope DOWNWARD from left to right.

Examine this dose-response curve and provide a structured quality assessment.
Respond in exactly this format:

QUALITY: [Good / Acceptable / Poor]
IC50_RELIABLE: [Yes / No / Uncertain]
ISSUES: [comma-separated list of issues, or 'None']
REASONING: [2-3 sentences explaining your assessment]
RECOMMENDATION: [one of: Accept / Flag for repeat / Reject]

Consider: sigmoid completeness, curve shape anomalies (hook effect, biphasic),
noise level, plateau definition, and whether the IC50 falls within the
tested concentration range."""

    response = client.messages.create(
        model='claude-sonnet-4-5',
        max_tokens=400,
        messages=[{
            'role': 'user',
            'content': [
                {'type': 'image',
                 'source': {'type': 'base64',
                            'media_type': 'image/png',
                            'data': img_b64}},
                {'type': 'text', 'text': prompt},
            ]
        }]
    )
    return response.content[0].text


# ── GPT-4o (commented out -- uncomment and comment the Claude block above) ──
#
# def assess_curve_gpt(name, df, fit, color, client):
#     img_b64 = render_curve_for_llm(name, df, fit, color)
#     prompt = """You are a medicinal chemist reviewing dose-response curves.
# Examine this curve and respond in exactly this format:
# QUALITY: [Good / Acceptable / Poor]
# IC50_RELIABLE: [Yes / No / Uncertain]
# ISSUES: [comma-separated list, or 'None']
# REASONING: [2-3 sentences]
# RECOMMENDATION: [Accept / Flag for repeat / Reject]"""
#     response = client.chat.completions.create(
#         model='gpt-4o',
#         max_tokens=400,
#         messages=[{'role': 'user', 'content': [
#             {'type': 'image_url',
#              'image_url': {'url': f'data:image/png;base64,{img_b64}'}},
#             {'type': 'text', 'text': prompt},
#         ]}]
#     )
#     return response.choices[0].message.content


print('LLM assessment functions defined.')

In [ ]:
# Initialise Claude client
claude_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# Uncomment to use GPT-4o instead:
# from openai import OpenAI
# gpt_client = OpenAI(api_key=OPENAI_API_KEY)

print('Client initialised.')

In [ ]:
# Run LLM assessment on all five compounds
colors = ['#0D9488', '#1B2A4A', '#E57373', '#F59E0B', '#6B7280']
llm_assessments = {}

for (name, df), color in zip(data.items(), colors):
    print(f'Assessing: {name}')
    assessment = assess_curve_claude(
        name, df, fit_results[name], color, claude_client
    )
    # Uncomment for GPT-4o:
    # assessment = assess_curve_gpt(name, df, fit_results[name], color, gpt_client)
    llm_assessments[name] = assessment
    print(assessment)
    print('-' * 60)

## 10. Side-by-side comparison: numerical QC vs LLM assessment

In [ ]:
def parse_llm_field(text, field):
    """Extract a single field value from the structured LLM response."""
    for line in text.split('\n'):
        if line.startswith(field + ':'):
            return line.split(':', 1)[1].strip()
    return 'N/A'


comparison_rows = []
for name, fit in fit_results.items():
    assessment = llm_assessments.get(name, '')
    row = {
        'Compound':         name,
        'R-squared':        round(fit['r_squared'], 3) if fit['success'] else 'N/A',
        'RMSE (%)':         round(fit['rmse'], 2)      if fit['success'] else 'N/A',
        'LLM Quality':      parse_llm_field(assessment, 'QUALITY'),
        'IC50 Reliable':    parse_llm_field(assessment, 'IC50_RELIABLE'),
        'Issues':           parse_llm_field(assessment, 'ISSUES'),
        'Recommendation':   parse_llm_field(assessment, 'RECOMMENDATION'),
    }
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

## 11. Final combined visualisation: curve + fit + LLM verdict

In [ ]:
verdict_colors = {'Good': '#0D9488', 'Acceptable': '#F59E0B', 'Poor': '#E57373'}

fig, axes = plt.subplots(2, 5, figsize=(17, 7),
                          gridspec_kw={'height_ratios': [3, 1]})
colors = ['#0D9488', '#1B2A4A', '#E57373', '#F59E0B', '#6B7280']

for col, (name, df), color in zip(range(5), data.items(), colors):
    ax_curve  = axes[0][col]
    ax_text   = axes[1][col]
    fit       = fit_results[name]
    assessment = llm_assessments.get(name, '')

    # --- Curve panel ---
    plot_fitted_curve(ax_curve, name, df, fit, color)

    # --- LLM verdict panel ---
    ax_text.axis('off')
    quality = parse_llm_field(assessment, 'QUALITY')
    rec     = parse_llm_field(assessment, 'RECOMMENDATION')
    issues  = parse_llm_field(assessment, 'ISSUES')
    vc      = verdict_colors.get(quality, '#888888')

    verdict_text = f'Quality: {quality}\n{rec}\n{issues[:40]}'
    ax_text.text(0.5, 0.5, verdict_text,
                 ha='center', va='center', fontsize=8,
                 color=vc, fontweight='bold',
                 transform=ax_text.transAxes,
                 bbox=dict(boxstyle='round,pad=0.3',
                           facecolor=vc + '22',
                           edgecolor=vc, linewidth=1.2))

fig.suptitle('Dose-Response Analysis: Fit + LLM Quality Assessment',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 12. Pointer: real dose-response data from ChEMBL

Everything above used synthetic data so we could control ground truth.
In real work, dose-response data comes from databases like ChEMBL.

The cell below shows how to query ChEMBL for a specific target and pull
IC50 measurements. We will use this data properly in Video 3 onwards.
For now, just run it to see the structure.

In [ ]:
!pip install chembl-webresource-client -q

In [ ]:
from chembl_webresource_client.new_client import new_client

# Query ChEMBL for EGFR (a well-studied cancer target)
# ChEMBL target ID for human EGFR: CHEMBL203
activity = new_client.activity

egfr_data = activity.filter(
    target_chembl_id='CHEMBL203',
    standard_type='IC50',
    standard_units='nM',
).only([
    'molecule_chembl_id',
    'standard_value',
    'standard_units',
    'assay_chembl_id',
])[:20]   # limit to 20 records for this demo

egfr_df = pd.DataFrame(list(egfr_data))
egfr_df['standard_value'] = pd.to_numeric(egfr_df['standard_value'],
                                            errors='coerce')
print(f'Retrieved {len(egfr_df)} EGFR IC50 records from ChEMBL')
egfr_df.head(10)

---
## Summary

In this notebook we:

- Built a 4PL dose-response model from scratch and explored how each parameter shapes the curve
- Generated five realistic synthetic scenarios (clean, noisy, hook effect, incomplete, inactive)
- Fitted each curve using `scipy.optimize.curve_fit` with data-driven initial guesses
- Extracted IC50 values with 95% confidence intervals and goodness-of-fit metrics
- Sent each curve to Claude Sonnet for structured quality assessment
- Compared numerical QC metrics against LLM judgement

**Key takeaways:**
- R-squared alone is not enough -- a hook effect curve can have a good R-squared if the wrong model is fitted
- CI width on IC50 is the most informative single number for reliability
- LLMs can catch qualitative issues (hook shape, incomplete plateau) that numerical metrics miss
- Numerical and LLM assessments are complementary, not alternatives

**Next:** Video 3 -- Molecular Representations (SMILES, fingerprints, descriptors with RDKit)